In [1]:
!pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 55.5 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44285 sha256=c871b0d99693f6391c5d6713da68618183073f76f8a87958eea1d102040906c0
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [2]:
import os
import gc
import copy
import torch
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

import timm
from torch import nn
from torch.optim import Adam
from torchvision import models
import torch.nn.functional as F
from torchvision.transforms import v2
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from typing import Callable,Tuple, List
from PIL import Image

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

In [3]:
# Check accelerator

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print(f'Using {device} device')

Using cuda device


In [4]:
# Set SEED for reproducibility

SEED = 24520152

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [5]:
# Train directory path

TRAIN_DIR = '/kaggle/input/datasets/hophamsailam/5-fold-brain-tumor-contrast-enhanced/kfold_dataset'
TRAIN_DIR

'/kaggle/input/datasets/hophamsailam/5-fold-brain-tumor-contrast-enhanced/kfold_dataset'

In [6]:
# Save directory path

SAVE_DIR = '/kaggle/working/'
SAVE_DIR

'/kaggle/working/'

In [7]:
# 3 labels: Glioma Tumor, Meningioma Tumor, Pituitary Tumor
CLASS_NAMES = sorted([d for d in os.listdir(os.path.join(TRAIN_DIR, 'Subset_1')) if os.path.isdir(os.path.join(TRAIN_DIR, 'Subset_1', d))])
CLASS_NAMES

['Glioma Tumor', 'Meningioma Tumor', 'Pituitary Tumor']

In [8]:
# Hyperparameters

BATCH_SIZE = 32
EPOCHS = 100
NUM_CLASSES = 3
DROPOUT_RATE = 0.3
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

In [9]:
MODEL_WEIGHTS = '/kaggle/input/datasets/hophamsailam/mobilevit-xxs-weights/student_distilled.pth'

In [10]:
def get_val_loader_for_fold(k: int, train_dir: str = TRAIN_DIR, batch_size: int = BATCH_SIZE) -> DataLoader:
    """
    Creates PyTorch Validation DataLoader for K-Fold Cross-Validation, specifically tailored
    for transfer learning with ImageNet standards.

    Args:
        k (int): The index of the validation fold (e.g., 1 to 5).
        train_dir (str): Path to the root directory containing the fold subsets 
                         (e.g., 'Subset_1', 'Subset_2', etc.).
        batch_size (int, optional): Number of samples per batch. Defaults to 32.

    Returns:
        val_loader (DataLoader): The validation data loader (not shuffled).
    """

    # Standard ImageNet normalization statistics
    norm_mean=[0.485, 0.456, 0.406]
    norm_std=[0.229, 0.224, 0.225]

    # Validation Transform Pipeline
    val_transform = v2.Compose([
        v2.Resize(size=256),
        v2.CenterCrop(size=224),
        v2.ToImage(),
        v2.ToDtype(dtype=torch.float32, scale=True),
        v2.Normalize(mean=norm_mean, std=norm_std)
    ])

    # Data Path Setup
    val_dir = os.path.join(train_dir, f'Subset_{k}')

    # Worker Configuration
    # Determine the optimal number of CPU workers to prevent bottlenecks.
    # Capped at 4 to avoid excessive memory overhead.
    num_workers = min(4, os.cpu_count())

    # Validation Loader
    val_dataset = ImageFolder(root=val_dir, transform=val_transform)
    val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    return val_loader

In [11]:
def build_student_model(num_classes: int = 3) -> nn.Module:
    """Builds a lightweight Student Model (MobileViT-XXS) for distillation.
    
    This function instantiates the Extra-Extra-Small (XXS) variant of MobileViT 
    using the `timm` library. MobileViT is a hybrid architecture that seamlessly 
    combines the spatial inductive biases of Convolutional Neural Networks (CNNs) 
    with the global attention mechanisms of Vision Transformers (ViTs).

    Args:
        num_classes (int, optional): Number of output classes for the 
            classification head. Defaults to 3.

    Returns:
        nn.Module: The initialized MobileViT-XXS PyTorch model (~1.2M params).
    """
    print("Initializing MobileViT-XXS student model...")
    
    # The timm library automatically downloads the pre-trained ImageNet weights 
    # and safely replaces the final classification head to match `num_classes`.
    model = timm.create_model(
        model_name='mobilevit_xxs', 
        pretrained=True, 
        num_classes=num_classes
    )
    
    return model

In [12]:
def visualize_gradcam(model: nn.Module, img_tensor: torch.Tensor, img_numpy: np.ndarray, true_class: int) -> np.ndarray:
    """
    Generates and visualizes Grad-CAM heatmaps for MobileViT-XXS.

    Args:
        model (nn.Module): The trained MobileViT-XXS model.
        img_tensor (torch.Tensor): Preprocessed image tensor with shape (1, 3, H, W).
        img_numpy (np.ndarray): Original image as a numpy array with shape (H, W, 3), scaled to [0, 1].
        true_class (int): ground truth label for img_tensor.
    Returns:
        np.ndarray: An array containing the Grad-CAM visual results.
    """
    model.eval()

    target_layers = [model.stages[-1]]
    targets = [ClassifierOutputTarget(true_class)]

    with GradCAM(model=model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]
        visualization = show_cam_on_image(img_numpy, grayscale_cam, use_rgb=True)

    return visualization

In [13]:
def generate_and_save_all_gradcams(fold_idx: int) -> None:
    model = build_student_model()
    model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=device, weights_only=True))
    model.eval()

    val_loader = get_val_loader_for_fold(k=fold_idx)
    dataset = val_loader.dataset

    GRADCAM_DIR = os.path.join(SAVE_DIR, f'GradCAM_Results_Fold_{fold_idx}')
    os.makedirs(GRADCAM_DIR, exist_ok=True)
    for class_name in CLASS_NAMES:
        os.makedirs(os.path.join(GRADCAM_DIR, class_name), exist_ok=True)

    plt.ioff()

    global_idx = 0

    for images, labels in tqdm(val_loader, desc='Generating Grad-CAMs'):
        images = images.to(device)

        for i in range(images.size(0)):
            img_tensor = images[i].unsqueeze(0).to(device) # Shape: (1, 3, 224, 224)
            img_tensor.requires_grad = True

            file_path = dataset.samples[global_idx][0]
            file_name = os.path.basename(file_path)

            img = Image.open(file_path).convert('RGB')
            img_numpy = np.array(img.resize((224, 224)))
            img_numpy = (img_numpy - img_numpy.min()) / (img_numpy.max() - img_numpy.min())

            true_label = labels[i].item()
            class_name = CLASS_NAMES[true_label]

            vis = visualize_gradcam(model=model, img_tensor=img_tensor, img_numpy=img_numpy, true_class=true_label)

            fig, axs = plt.subplots(1, 2, figsize=(10, 5))
                
            axs[0].imshow(img_numpy)
            axs[0].set_title(f"Original (Class: {class_name})", fontsize=12, fontweight='bold')
            axs[0].axis('off')
            
            axs[1].imshow(vis)
            axs[1].set_title("MobileViT-XXS Grad-CAM", fontsize=12, fontweight='bold')
            axs[1].axis('off')
            
            plt.tight_layout()

            save_path = os.path.join(GRADCAM_DIR, class_name, file_name)
            plt.savefig(save_path, dpi=150, bbox_inches='tight')

            global_idx += 1

In [14]:
generate_and_save_all_gradcams(fold_idx=1)

Initializing MobileViT-XXS student model...


model.safetensors:   0%|          | 0.00/5.14M [00:00<?, ?B/s]

Generating Grad-CAMs:   0%|          | 0/17 [00:00<?, ?it/s]/tmp/ipykernel_55/1595998648.py:37: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axs = plt.subplots(1, 2, figsize=(10, 5))
Generating Grad-CAMs: 100%|██████████| 17/17 [04:08<00:00, 14.61s/it]


In [15]:
!zip -r -q 'MobileViT-XXS_GradCAM' '/kaggle/working/GradCAM_Results_Fold_1'